# Module 10 part2

# Part B — Propositional Logic & First-Order Logic (Conceptual)



The module briefly introduces logic for text analysis:

## Propositional Logic
Represents simple truth-valued statements such as:
- “The review is positive”
- “The feedback is negative”

Example rule:
- If text contains positive words **AND** lacks negative words, then sentiment is positive.

## First-Order Logic (FOL)
More expressive because it can represent:
- predicates,
- quantifiers,
- relationships between words/sentences.

Example:
- “For all sentences in the review, if a sentence contains *excellent*, then it is positive.”

These ideas are useful conceptually for rule-based NLP systems.  



---

# Part C — Named Entity Recognition (NER)

**NER** identifies and classifies entities in text such as:
- PERSON
- ORG
- GPE (geopolitical entity / locations)
- DATE
- PRODUCT

The module uses **spaCy** with `en_core_web_sm`.  


In [41]:

import spacy

# Load spaCy English model
# Make sure: python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

text = '''
Apple was founded by Steve Jobs in Cupertino. Barack Obama was the 44th president of the United States.
The Eiffel Tower is located in Paris, France. Microsoft acquired LinkedIn in 2016.
'''

doc = nlp(text)

print("Named Entities:")
for ent in doc.ents:
    print(f"{ent.text:20s} -> {ent.label_}")


Named Entities:
Apple                -> ORG
Steve Jobs           -> PERSON
Cupertino            -> GPE
Barack Obama         -> PERSON
44th                 -> ORDINAL
the United States    -> GPE
The Eiffel Tower     -> LOC
Paris                -> GPE
France               -> GPE
Microsoft            -> ORG
LinkedIn             -> ORG
2016                 -> DATE



## . Optional: Display NER in a Rich Visual Format


In [42]:

# Optional visualization in Jupyter
# from spacy import displacy
# displacy.render(doc, style="ent", jupyter=True)



### Interpretation
Examples you should see:
- **Apple** → ORG
- **Steve Jobs** → PERSON
- **Cupertino** → GPE
- **Barack Obama** → PERSON
- **United States** → GPE
- **Microsoft** → ORG
- **LinkedIn** → ORG
- **2016** → DATE



---

# Part D — Sentiment Analysis

Sentiment analysis determines whether text is:
- **positive**
- **negative**
- **neutral**

The module introduces two approaches:
1. **Lexicon-based** (VADER)
2. **Machine learning / transformer-based** (Hugging Face)  



## . Sentiment Analysis with VADER

VADER is:
- rule-based,
- lexicon-based,
- especially good for short informal text (social media, reviews, emojis, punctuation emphasis).

The PDF notes that VADER handles:
- punctuation emphasis (`!!!`)
- capitalization
- negation words like `not`
- contrast words like `but`


In [43]:
#!pip install --upgrade ipywidgets notebook jupyterlab

In [44]:
#!pip install vaderSentiment

In [45]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

samples = [
    "I love this phone! It's amazing!!!",
    "This product is okay, nothing special.",
    "I hate this update. It is terrible."
]

for s in samples:
    scores = analyzer.polarity_scores(s)
    print(f"Text: {s}")
    print(scores)
    print("-" * 70)

Text: I love this phone! It's amazing!!!
{'neg': 0.0, 'neu': 0.304, 'pos': 0.696, 'compound': 0.8798}
----------------------------------------------------------------------
Text: This product is okay, nothing special.
{'neg': 0.277, 'neu': 0.49, 'pos': 0.233, 'compound': -0.092}
----------------------------------------------------------------------
Text: I hate this update. It is terrible.
{'neg': 0.576, 'neu': 0.424, 'pos': 0.0, 'compound': -0.7783}
----------------------------------------------------------------------



### VADER Output Meaning
- `neg`: proportion of negative sentiment
- `neu`: proportion of neutral sentiment
- `pos`: proportion of positive sentiment
- `compound`: normalized overall score in **[-1, 1]**

A simple rule of thumb:
- `compound >= 0.05` → positive
- `compound <= -0.05` → negative
- otherwise → neutral


In [46]:

def vader_label(compound):
    if compound >= 0.05:
        return "POSITIVE"
    elif compound <= -0.05:
        return "NEGATIVE"
    else:
        return "NEUTRAL"

for s in samples:
    scores = analyzer.polarity_scores(s)
    print(f"{s} -> {vader_label(scores['compound'])} (compound={scores['compound']:.4f})")


I love this phone! It's amazing!!! -> POSITIVE (compound=0.8798)
This product is okay, nothing special. -> NEGATIVE (compound=-0.0920)
I hate this update. It is terrible. -> NEGATIVE (compound=-0.7783)



## . Sentiment Analysis with Hugging Face Transformers

The module highlights that Hugging Face provides a **simple API** to load powerful pretrained models for sentiment analysis.  


The Medium article also emphasizes that the `pipeline()` function is the **easiest way to get started** with Hugging Face Transformers and demonstrates sentiment analysis as a beginner-friendly example.


In [47]:

from transformers import pipeline

# Default sentiment-analysis pipeline
# Often defaults to a DistilBERT-based SST-2 classifier
classifier = pipeline("sentiment-analysis")

inputs = [
    "The beautiful thing with learning is that no one can take it away from you!",
    "Hugging Face is cool.",
    "I want problems, always."
]

results = classifier(inputs)

for text, result in zip(inputs, results):
    print(f"Text: {text}")
    print(f"Label: {result['label']}, Score: {result['score']:.4f}")
    print("-" * 70)


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: The beautiful thing with learning is that no one can take it away from you!
Label: POSITIVE, Score: 0.9985
----------------------------------------------------------------------
Text: Hugging Face is cool.
Label: POSITIVE, Score: 0.9999
----------------------------------------------------------------------
Text: I want problems, always.
Label: NEGATIVE, Score: 0.9923
----------------------------------------------------------------------



### Important Note
The default `sentiment-analysis` pipeline may map labels like:
- `POSITIVE`
- `NEGATIVE`

Depending on the model, labels and domains may differ. In real projects, it is often better to specify the model explicitly.



---

# Part E — Hugging Face Transformers: Practical Introduction

The Medium article is a strong beginner-friendly supplement to the PDF.

## Key Takeaways from the Article
- Hugging Face Transformers provides **pretrained state-of-the-art models** for many ML tasks.
- It is built around the **transformer architecture** and attention mechanisms.
- It supports multiple tasks with a very simple `pipeline()` interface.
- It is especially useful for:
  - rapid prototyping,
  - text classification,
  - summarization,
  - question answering,
  - content generation,
  - even multimodal tasks like audio classification.

The official Hugging Face docs also confirm that Transformers supports **NLP, computer vision, audio, and speech tasks**, not only text.



## . Why `pipeline()` is So Popular

`pipeline()` hides many of the complex steps:
- tokenizer loading,
- model loading,
- preprocessing,
- inference,
- output formatting.




## . Additional Hugging Face Example: Question Answering

This example is inspired by the Medium article’s QA demo.  
The article uses `pipeline("question-answering")` with a simple context-question pair.


In [48]:

qa_model = pipeline("question-answering")

context = "Hugging Face was founded in 2016 as a chatbot for teenagers."
question = "When was Hugging Face founded?"

answer = qa_model(question=question, context=context)

print("Question:", question)
print("Answer:", answer["answer"])
print("Confidence:", round(answer["score"], 4))


No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Question: When was Hugging Face founded?
Answer: 2016
Confidence: 0.9794



## . Additional Hugging Face Example: Zero-Shot Classification (Bonus)

This is **not in the PDF**, but it is a great extension for a modern NLP lecture.


In [49]:

# Optional / bonus
# This model may be larger and take longer to load.
# zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
# sequence = "This course module explains document summarization, NER, and sentiment analysis."
# labels = ["education", "sports", "finance", "politics"]
# result = zero_shot(sequence, labels)
# pprint(result)



---

# Part F — Mini Comparison: VADER vs Hugging Face


In [50]:

comparison_texts = [
    "I absolutely love this camera. Best purchase ever!",
    "The camera is fine. Battery life is average.",
    "Worst camera I have used. Total waste of money."
]

print("=== VADER ===")
for t in comparison_texts:
    sc = analyzer.polarity_scores(t)
    print(f"{t}\n -> {vader_label(sc['compound'])}, compound={sc['compound']:.4f}\n")

print("=== Hugging Face ===")
hf_results = classifier(comparison_texts)
for t, r in zip(comparison_texts, hf_results):
    print(f"{t}\n -> {r['label']}, score={r['score']:.4f}\n")


=== VADER ===
I absolutely love this camera. Best purchase ever!
 -> POSITIVE, compound=0.8746

The camera is fine. Battery life is average.
 -> POSITIVE, compound=0.2023

Worst camera I have used. Total waste of money.
 -> NEGATIVE, compound=-0.8016

=== Hugging Face ===
I absolutely love this camera. Best purchase ever!
 -> POSITIVE, score=0.9999

The camera is fine. Battery life is average.
 -> POSITIVE, score=0.8095

Worst camera I have used. Total waste of money.
 -> NEGATIVE, score=0.9998




## Discussion: Which one should you use?

### Use **VADER** when:
- text is short,
- text is informal (tweets, comments),
- you need something fast/lightweight,
- you want no model download.

### Use **Hugging Face Transformers** when:
- you need stronger semantic understanding,
- domain accuracy matters more,
- you can afford model download/inference time,
- you want easy access to modern pretrained models.
